In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../'))

In [2]:
from neuro_fuzzy_toolbox_old import Type3ANFIS
from neuro_fuzzy_toolbox_old import hybrid_algorithm, EarlyStopping

In [3]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Data

In [4]:
data_df = pd.read_csv('data/healthcare-dataset-stroke-data.csv')
data_df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [5]:
data_df['gender'] = data_df['gender'].map({'Male': 1, 'Female': 0})
data_df['ever_married'] = data_df['ever_married'].map({'Yes': 1, 'No': 0})
data_df = pd.get_dummies(data_df, columns=['work_type'], prefix='work')
data_df['Residence_type'] = data_df['Residence_type'].map({'Urban': 1, 'Rural': 0})
data_df = pd.get_dummies(data_df, columns=['smoking_status'], prefix='smoke')
data_df.fillna(0, inplace=True)
data_df

,id,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,work_Govt_job,work_Never_worked,work_Private,work_Self-employed,work_children,smoke_Unknown,smoke_formerly smoked,smoke_never smoked,smoke_smokes
0,9046,1.0,67.0,0,1,1,1,228.69,36.6,1,False,False,True,False,False,False,True,False,False
1,51676,0.0,61.0,0,0,1,0,202.21,0.0,1,False,False,False,True,False,False,False,True,False
2,31112,1.0,80.0,0,1,1,0,105.92,32.5,1,False,False,True,False,False,False,False,True,False
3,60182,0.0,49.0,0,0,1,1,171.23,34.4,1,False,False,True,False,False,False,False,False,True
4,1665,0.0,79.0,1,0,1,0,174.12,24.0,1,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,0.0,80.0,1,0,1,1,83.75,0.0,0,False,False,True,False,False,False,False,True,False
5106,44873,0.0,81.0,0,0,1,1,125.20,40.0,0,False,False,False,True,False,False,False,True,False
5107,19723,0.0,35.0,0,0,1,0,82.99,30.6,0,False,False,False,True,False,False,False,True,False
5108,37544,1.0,51.0,0,0,1,0,166.29,25.6,0,False,False,True,False,False,False,True,False,False


In [6]:
X = data_df.drop(columns=['id', 'heart_disease'])
Y = data_df['heart_disease']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [7]:
scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

Y_train_tensor = torch.tensor(Y_train.values, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype=torch.float32)

In [9]:
X_train_tensor.shape, Y_train_tensor.shape

(torch.Size([4088, 17]), torch.Size([4088]))

In [10]:
loader = data.DataLoader(data.TensorDataset(X_train_tensor, Y_train_tensor), batch_size = 16, shuffle = True)

# Model

In [11]:
model = Type3ANFIS(X_train_tensor, init_rules=20, output_type="binary")
model.fuzzify_layer.premises.shape, model.consequent_layer.consequents.shape

(torch.Size([17, 20, 2]), torch.Size([1, 20, 18]))

In [12]:
epochs = 20
optimizer = torch.optim.Adam
params = {'lr': 0.001, 'weight_decay': 0.001}

early_stopping = EarlyStopping(patience=7, delta=0.01)

In [13]:
algorithm = hybrid_algorithm(epochs=epochs,
                             loss_function=nn.functional.binary_cross_entropy,
                             optimizer=optimizer,
                             optim_params=params,
                             validation=0.2,
                             early_stopping=early_stopping)

In [14]:
algorithm(model, loader)

/home/jsuarez/workspaces/neuro-fuzzy-toolbox/neuro_fuzzy_toolbox_old/training/hybrid_algorithm.py:217: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  self.val_history[measure] =  torch.cat([self.val_history[measure], torch.tensor([measures[measure]])])


RuntimeError: all elements of input should be between 0 and 1